<a href="https://colab.research.google.com/github/aendris18/Frist-class/blob/main/multiclassml%20dp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ============================================================
# Step 1: Environment Setup and Data Loading
# Objective 1: Prepare benchmark network intrusion dataset
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os

warnings.filterwarnings('ignore')

# ============================================================
# Machine Learning Libraries
# ============================================================

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score
)

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from imblearn.over_sampling import SMOTE

# ============================================================
# Deep Learning Libraries
# ============================================================

import tensorflow as tf

from tensorflow.keras.models import Model, Sequential

from tensorflow.keras.layers import (
    Dense,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dropout,
    BatchNormalization,
    Input,
    Flatten,
    concatenate
)

from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

# ============================================================
# Check TensorFlow
# ============================================================

print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

# ============================================================
# Load UNSW-NB15 Dataset
# ============================================================

# Your dataset folder
DATA_DIR = r"/content/"

# Dataset files
TRAIN_PATH = os.path.join(
    DATA_DIR,
    "UNSW_NB15_training-set.csv"
)

TEST_PATH = os.path.join(
    DATA_DIR,
    "UNSW_NB15_testing-set.csv"
)

# ============================================================
# Check whether files exist
# ============================================================

print("\n" + "=" * 60)
print("DATASET PATH CHECK")
print("=" * 60)

print("Dataset directory:", DATA_DIR)
print("Training file:", TRAIN_PATH)
print("Testing file:", TEST_PATH)

print("\nTraining file exists:", os.path.exists(TRAIN_PATH))
print("Testing file exists:", os.path.exists(TEST_PATH))

# Stop with a clear error if files are not found
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        f"\nTraining dataset not found:\n{TRAIN_PATH}"
    )

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(
        f"\nTesting dataset not found:\n{TEST_PATH}"
    )

# ============================================================
# Read CSV files
# ============================================================

df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

# ============================================================
# Display Dataset Information
# ============================================================

print("\n" + "=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print(f"Training set shape: {df_train.shape}")
print(f"Testing set shape:  {df_test.shape}")

print("\nTraining set columns:")
print(list(df_train.columns))

print("\nTesting set columns:")
print(list(df_test.columns))

# ============================================================
# Attack Category Distribution
# ============================================================

print("\n" + "=" * 60)
print("ATTACK CATEGORY DISTRIBUTION - TRAINING SET")
print("=" * 60)

print(df_train["attack_cat"].value_counts())

print("\n" + "=" * 60)
print("ATTACK CATEGORY DISTRIBUTION - TESTING SET")
print("=" * 60)

print(df_test["attack_cat"].value_counts())

# ============================================================
# Basic Dataset Information
# ============================================================

print("\n" + "=" * 60)
print("FIRST 5 TRAINING RECORDS")
print("=" * 60)

display(df_train.head())

print("\n" + "=" * 60)
print("DATA TYPES")
print("=" * 60)

print(df_train.dtypes)

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

print("Training missing values:")
print(df_train.isnull().sum())

print("\nTesting missing values:")
print(df_test.isnull().sum())

ENVIRONMENT INFORMATION
TensorFlow version: 2.20.0
GPU available: []

DATASET PATH CHECK
Dataset directory: /content/
Training file: /content/UNSW_NB15_training-set.csv
Testing file: /content/UNSW_NB15_testing-set.csv

Training file exists: True
Testing file exists: True

DATASET INFORMATION
Training set shape: (82332, 45)
Testing set shape:  (175341, 45)

Training set columns:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'attack_cat', 'label']

Testing set columns:
['id', 'dur', 'proto', 'service', 'state', 'spkts', 'dpkts', 'sbytes', 'dbytes',

,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0



DATA TYPES
id                     int64
dur                  float64
proto                 object
service               object
state                 object
spkts                  int64
dpkts                  int64
sbytes                 int64
dbytes                 int64
rate                 float64
sttl                   int64
dttl                   int64
sload                float64
dload                float64
sloss                  int64
dloss                  int64
sinpkt               float64
dinpkt               float64
sjit                 float64
djit                 float64
swin                   int64
stcpb                  int64
dtcpb                  int64
dwin                   int64
tcprtt               float64
synack               float64
ackdat               float64
smean                  int64
dmean                  int64
trans_depth            int64
response_body_len      int64
ct_srv_src             int64
ct_state_ttl           int64
ct_dst_ltm             int64
ct

In [9]:
# ============================================================
# Step 2: Data Cleaning and Preprocessing
# Objective 2: Data cleaning, categorical encoding, feature
#              selection, scaling, and class-imbalance handling
# ============================================================

# ------------------------------------------------------------
# 2.1 Data Cleaning
# ------------------------------------------------------------
def clean_data(df):
    """Clean the UNSW-NB15 dataset"""
    df = df.copy()

    # Standardize column names: strip spaces and lowercase
    df.columns = [c.strip().lower() for c in df.columns]

    # Handle '-' placeholder in 'service' column
    if 'service' in df.columns:
        df['service'] = df['service'].replace('-', np.nan)

    # Handle potential non-numeric issues in 'ct_ftp_cmd'
    if 'ct_ftp_cmd' in df.columns:
        df['ct_ftp_cmd'] = pd.to_numeric(df['ct_ftp_cmd'], errors='coerce')

    # Drop missing values
    before = df.shape[0]
    df.dropna(inplace=True)
    after = df.shape[0]
    print(f"  Missing values removed: {before} -> {after} ({before - after} rows dropped)")

    return df

print("Cleaning training set...")
df_train = clean_data(df_train)
print("Cleaning testing set...")
df_test = clean_data(df_test)

# Align columns between train and test
common_cols = list(set(df_train.columns) & set(df_test.columns))
df_train = df_train[common_cols]
df_test = df_test[common_cols]

# ------------------------------------------------------------
# 2.2 Attack Category Standardization
# ------------------------------------------------------------
print("\nOriginal attack category distribution:")
print(df_train['attack_cat'].value_counts())

# Merge low-frequency categories to mitigate extreme imbalance
rare_categories = ['Backdoor', 'Shellcode', 'Worms']
df_train['attack_cat'] = df_train['attack_cat'].replace(rare_categories, 'Rare')
df_test['attack_cat']  = df_test['attack_cat'].replace(rare_categories, 'Rare')

print("\nStandardized attack category distribution:")
print(df_train['attack_cat'].value_counts())

# ------------------------------------------------------------
# 2.3 Feature Type Identification and Separation
# ------------------------------------------------------------
TARGET_BINARY = 'label'
TARGET_MULTI = 'attack_cat'

# Exclude ID and target columns from features
drop_cols = ['id', TARGET_BINARY, TARGET_MULTI]
drop_cols = [c for c in drop_cols if c in df_train.columns]

feature_cols = [c for c in df_train.columns if c not in drop_cols]

# Identify numerical and categorical columns
cat_cols = df_train[feature_cols].select_dtypes(include=['object']).columns.tolist()
num_cols = df_train[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

print(f"\nNumerical features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"Categorical feature list: {cat_cols}")

# ------------------------------------------------------------
# 2.4 Label Encoding for Multi-Class Target
# ------------------------------------------------------------
le_attack = LabelEncoder()
# Fit on the combined unique labels from both train and test sets to avoid unseen labels in transform
all_attack_cats = pd.concat([df_train[TARGET_MULTI], df_test[TARGET_MULTI]]).unique()
le_attack.fit(all_attack_cats)

y_train_multi = le_attack.transform(df_train[TARGET_MULTI])
y_test_multi  = le_attack.transform(df_test[TARGET_MULTI])

n_classes = len(le_attack.classes_)
print(f"\nNumber of attack classes: {n_classes}")
print(f"Class mapping: {dict(zip(le_attack.classes_, range(n_classes)))}")

# Binary labels
y_train_bin = df_train[TARGET_BINARY].values
y_test_bin  = df_test[TARGET_BINARY].values

# ------------------------------------------------------------
# 2.5 Categorical Feature Encoding: One-Hot Encoding
# ------------------------------------------------------------
# Convert categorical columns to string for consistency
for c in cat_cols:
    df_train[c] = df_train[c].astype(str)
    df_test[c] = df_test[c].astype(str)

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_train[cat_cols])

train_cat_encoded = pd.DataFrame(
    ohe.transform(df_train[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=df_train.index
)
test_cat_encoded = pd.DataFrame(
    ohe.transform(df_test[cat_cols]),
    columns=ohe.get_feature_names_out(cat_cols),
    index=df_test.index
)

print(f"One-Hot encoded categorical dimension: {train_cat_encoded.shape[1]}")

# Merge numerical and encoded categorical features
X_train_raw = pd.concat([df_train[num_cols], train_cat_encoded], axis=1)
X_test_raw  = pd.concat([df_test[num_cols], test_cat_encoded], axis=1)

# Ensure column alignment
X_test_raw = X_test_raw.reindex(columns=X_train_raw.columns, fill_value=0)

print(f"Encoded training set shape: {X_train_raw.shape}")
print(f"Encoded testing set shape: {X_test_raw.shape}")

# ------------------------------------------------------------
# 2.6 Missing Value Imputation (post-encoding)
# ------------------------------------------------------------
X_train_raw = X_train_raw.fillna(0)
X_test_raw  = X_test_raw.fillna(0)

# ------------------------------------------------------------
# 2.7 Feature Selection: Mutual Information
# ------------------------------------------------------------
print("\nPerforming feature selection (mutual information)...")
selector = SelectKBest(score_func=mutual_info_classif, k='all')
selector.fit(X_train_raw, y_train_multi)

# Rank feature importance
feature_scores = pd.DataFrame({
    'feature': X_train_raw.columns,
    'mi_score': selector.scores_
}).sort_values('mi_score', ascending=False)

print("Top 20 most important features:")
print(feature_scores.head(20).to_string(index=False))

# Select top-k features
k_best = min(45, X_train_raw.shape[1])
top_features = feature_scores.head(k_best)['feature'].tolist()

X_train_selected = X_train_raw[top_features]
X_test_selected  = X_test_raw[top_features]

print(f"\nDimension after feature selection: {X_train_selected.shape[1]}")

# ------------------------------------------------------------
# 2.8 Feature Scaling: StandardScaler
# ------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_test_scaled  = scaler.transform(X_test_selected)

print(f"Scaled training set shape: {X_train_scaled.shape}")
print(f"Scaled testing set shape: {X_test_scaled.shape}")

# ------------------------------------------------------------
# 2.9 Class Imbalance Handling: SMOTE (training set only)
# ------------------------------------------------------------
print("\nOriginal training set class distribution:")
unique, counts = np.unique(y_train_multi, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  {le_attack.classes_[cls]}: {cnt}")

# Apply SMOTE oversampling on the training set
smote = SMOTE(random_state=42, k_neighbors=3)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train_multi)

print("\nTraining set class distribution after SMOTE:")
unique_res, counts_res = np.unique(y_train_res, return_counts=True)
for cls, cnt in zip(unique_res, counts_res):
    print(f"  {le_attack.classes_[cls]}: {cnt}")

print(f"\nTraining set shape after SMOTE: {X_train_res.shape}")
print(f"Testing set shape (unchanged): {X_test_scaled.shape}")

Cleaning training set...
  Missing values removed: 35179 -> 35179 (0 rows dropped)
Cleaning testing set...
  Missing values removed: 81173 -> 81173 (0 rows dropped)

Original attack category distribution:
attack_cat
Generic           18460
Normal             9625
Exploits           5293
DoS                 717
Fuzzers             535
Reconnaissance      504
Rare                 45
Name: count, dtype: int64

Standardized attack category distribution:
attack_cat
Generic           18460
Normal             9625
Exploits           5293
DoS                 717
Fuzzers             535
Reconnaissance      504
Rare                 45
Name: count, dtype: int64

Numerical features: 39
Categorical features: 3
Categorical feature list: ['service', 'state', 'proto']

Number of attack classes: 8
Class mapping: {'Analysis': 0, 'DoS': 1, 'Exploits': 2, 'Fuzzers': 3, 'Generic': 4, 'Normal': 5, 'Rare': 6, 'Reconnaissance': 7}
One-Hot encoded categorical dimension: 19
Encoded training set shape: (35179, 5

In [ ]:
# ============================================================
# Step 3: Baseline Machine Learning Models
# Objective 3: Develop baseline ML intrusion classifiers
# ============================================================

# ------------------------------------------------------------
# 3.1 Evaluation Function Definition
# ------------------------------------------------------------
def evaluate_model(model, X_test, y_test, model_name, le, n_classes):
    """
    Comprehensive model evaluation: accuracy, precision, recall,
    F1, Cohen's Kappa, and training/inference time.
    """
    # Inference time
    start_time = time.time()
    y_pred = model.predict(X_test)
    inference_time = time.time() - start_time

    # Overall metrics (weighted average)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_test, y_pred)

    # Per-class metrics
    per_class_prec = precision_score(y_test, y_pred, average=None, zero_division=0)
    per_class_rec = recall_score(y_test, y_pred, average=None, zero_division=0)
    per_class_f1 = f1_score(y_test, y_pred, average=None, zero_division=0)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    results = {
        'name': model_name,
        'accuracy': acc,
        'precision_weighted': prec,
        'recall_weighted': rec,
        'f1_weighted': f1,
        'kappa': kappa,
        'inference_time': inference_time,
        'per_class_precision': per_class_prec,
        'per_class_recall': per_class_rec,
        'per_class_f1': per_class_f1,
        'confusion_matrix': cm,
        'predictions': y_pred
    }

    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    print(f"Accuracy:                {acc:.4f}")
    print(f"Precision (weighted):    {prec:.4f}")
    print(f"Recall (weighted):       {rec:.4f}")
    print(f"F1-score (weighted):     {f1:.4f}")
    print(f"Cohen's Kappa:           {kappa:.4f}")
    print(f"Inference time:          {inference_time:.4f} sec")

    return results

def plot_confusion_matrix(cm, classes, title, ax=None):
    """Plot confusion matrix heatmap"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax,
                cbar_kws={'label': 'Sample count'})
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    return ax

def plot_per_class_metrics(results, classes):
    """Visualize per-class precision, recall, F1"""
    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(classes))
    width = 0.25

    ax.bar(x - width, results['per_class_precision'], width, label='Precision', color='#2ecc71')
    ax.bar(x, results['per_class_recall'], width, label='Recall', color='#3498db')
    ax.bar(x + width, results['per_class_f1'], width, label='F1-score', color='#e74c3c')

    ax.set_xlabel('Attack Class')
    ax.set_ylabel('Score')
    ax.set_title(f'{results["name"]} - Per-Class Performance Metrics')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim([0, 1.05])
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 3.2 Define Baseline Models
# ------------------------------------------------------------
baseline_models = {
    'Decision Tree (DT)': DecisionTreeClassifier(
        max_depth=15, min_samples_split=5, random_state=42
    ),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(
        n_neighbors=5, n_jobs=-1
    ),
    'Logistic Regression (LR)': LogisticRegression(
        max_iter=1000, multi_class='multinomial',
        solver='lbfgs', random_state=42, n_jobs=-1
    ),
    'Support Vector Machine (SVM)': SVC(
        kernel='rbf', C=1.0, gamma='scale',
        probability=False, random_state=42
    )
}

# ------------------------------------------------------------
# 3.3 Train and Evaluate Baseline Models
# ------------------------------------------------------------
baseline_results = {}

for name, model in baseline_models.items():
    print(f"\nTraining {name}...")

    # Training time
    train_start = time.time()
    model.fit(X_train_res, y_train_res)
    train_time = time.time() - train_start

    # Evaluate
    result = evaluate_model(model, X_test_scaled, y_test_multi, name, le_attack, n_classes)
    result['train_time'] = train_time

    baseline_results[name] = result

    print(f"Training time: {train_time:.2f} sec")

# ------------------------------------------------------------
# 3.4 Baseline Model Comparison Visualization
# ------------------------------------------------------------
print("\n" + "="*80)
print("Baseline Model Performance Comparison")
print("="*80)

comparison_df = pd.DataFrame({
    'Model': [r['name'] for r in baseline_results.values()],
    'Accuracy': [r['accuracy'] for r in baseline_results.values()],
    'Precision(w)': [r['precision_weighted'] for r in baseline_results.values()],
    'Recall(w)': [r['recall_weighted'] for r in baseline_results.values()],
    'F1(w)': [r['f1_weighted'] for r in baseline_results.values()],
    'Kappa': [r['kappa'] for r in baseline_results.values()],
    'Train Time(s)': [r['train_time'] for r in baseline_results.values()],
    'Inference Time(s)': [r['inference_time'] for r in baseline_results.values()]
})

print(comparison_df.to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['accuracy', 'f1_weighted', 'kappa']
labels = ['Accuracy', 'F1(weighted)', 'Kappa']
x = np.arange(len(baseline_results))
width = 0.25

for i, (metric, label) in enumerate(zip(metrics, labels)):
    values = [r[metric] for r in baseline_results.values()]
    axes[0].bar(x + i*width, values, width, label=label)

axes[0].set_xlabel('Model')
axes[0].set_ylabel('Score')
axes[0].set_title('Baseline Model Overall Performance')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels([r['name'] for r in baseline_results.values()], rotation=15)
axes[0].legend()
axes[0].set_ylim([0, 1.05])
axes[0].grid(axis='y', alpha=0.3)

# Time comparison
train_times = [r['train_time'] for r in baseline_results.values()]
infer_times = [r['inference_time'] for r in baseline_results.values()]

axes[1].bar(x - width/2, train_times, width, label='Training time', color='#e67e22')
axes[1].bar(x + width/2, infer_times, width, label='Inference time', color='#9b59b6')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Time (sec)')
axes[1].set_title('Baseline Model Training and Inference Time')
axes[1].set_xticks(x)
axes[1].set_xticklabels([r['name'] for r in baseline_results.values()], rotation=15)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Best baseline model
best_baseline_name = max(baseline_results, key=lambda k: baseline_results[k]['f1_weighted'])
print(f"\nBest baseline model: {best_baseline_name}")
print(f"  Accuracy: {baseline_results[best_baseline_name]['accuracy']:.4f}")
print(f"  F1(weighted): {baseline_results[best_baseline_name]['f1_weighted']:.4f}")


Training Decision Tree (DT)...

Model: Decision Tree (DT)
Accuracy:                0.8920
Precision (weighted):    0.9134
Recall (weighted):       0.8920
F1-score (weighted):     0.8973
Cohen's Kappa:           0.8390
Inference time:          0.0176 sec
Training time: 18.60 sec

Training K-Nearest Neighbors (KNN)...

Model: K-Nearest Neighbors (KNN)
Accuracy:                0.8776
Precision (weighted):    0.8934
Recall (weighted):       0.8776
F1-score (weighted):     0.8828
Cohen's Kappa:           0.8174
Inference time:          81.2611 sec
Training time: 0.01 sec

Training Logistic Regression (LR)...

Model: Logistic Regression (LR)
Accuracy:                0.8571
Precision (weighted):    0.9138
Recall (weighted):       0.8571
F1-score (weighted):     0.8704
Cohen's Kappa:           0.7902
Inference time:          0.0434 sec
Training time: 65.35 sec

Training Support Vector Machine (SVM)...
